# When Does Test-Time Prompt Grouping Help?

In authorship attribution, every passage answers some prompt, and the prompt leaves its own mark on the text. If
several test passages answer the same prompt, a decoder can remove that shared mark (centring), or forbid two of them
from getting the same author (Hungarian assignment). A grouped cross-validation on a train set where every prompt is
answered by every author promises large gains from this. On the public leaderboard of *Name That LLM*, the same tricks
gained much less or lost (one forum report: +0.10 in grouped CV, then 0.353 to 0.344 on the public board).

This notebook simulates the problem with a known truth: 18 authors in 6 families, a shared prompt effect, and test pools
with a controlled number of answers per prompt. It measures what each decoder gains when the groups are true, when they
are recovered by clustering, and when author frequencies are uneven.

The decoders:

- `plain`: each passage on its own.
- `center`: subtract the group mean of the scores, then decode as if the prompt effect were gone (the common heuristic).
- `hungarian`: plain scores, then at most one passage per author inside each group.
- `joint`: decode each member against the prompt effect estimated from the other members (iterated conditional modes
  under the true model).
- `joint_no_repeat`: `joint` plus the one-passage-per-author constraint.
- `plain_sinkhorn`: plain scores, then rescale until every author holds the same share of the probability mass.

Everything runs on CPU in about 10 to 15 minutes. The package source is embedded in the next cell, so the notebook is
self-contained.


In [ ]:
SOURCES = {
"pgbench/__init__.py": "\"\"\"Known-truth simulation of test-time prompt grouping for authorship attribution.\"\"\"\n",
"pgbench/__main__.py": "\"\"\"python -m pgbench [outdir]: run every experiment, print the summaries and save the raw tables.\"\"\"\n\nimport sys\nimport time\nfrom pathlib import Path\n\nfrom . import experiments as ex\n\n\ndef main(out: Path) -> None:\n    out.mkdir(parents=True, exist_ok=True)\n    for name, fn, summary in [(\"exp1_layouts\", ex.layouts, ex.summarise_layouts),\n                              (\"exp2_grouping\", ex.grouping, ex.summarise_grouping),\n                              (\"exp3_prior_shift\", ex.prior_shift, ex.summarise_prior),\n                              (\"exp4_promise\", ex.promise, ex.summarise_promise)]:\n        t = time.time()\n        df = fn()\n        df.to_csv(out / f\"{name}.csv\", index=False)\n        print(f\"\\n== {name} ({time.time() - t:.0f}s)\\n{summary(df).to_string(index=False)}\", flush=True)\n\n\nif __name__ == \"__main__\":\n    main(Path(sys.argv[1] if len(sys.argv) > 1 else \"results\"))\n",
"pgbench/experiments.py": "\"\"\"The four experiments. Each returns one row per (setting, replicate, decoder).\"\"\"\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom .methods import DECODERS, cluster, oracle, pair_quality, singletons, sinkhorn, _loglik\nfrom .sim import K, World, balanced_accuracy, layout, sample, skewed_prior, test_pool\n\nSEED = 20261021\nN_TEST = 4000\nLAYOUTS = [1, 2, 3, 6, 12, 18, \"mixed\"]\n\n\ndef _score(pool, groups, w, decoders=DECODERS):\n    return {name: balanced_accuracy(pool.labels, f(pool.scores, groups, w)) for name, f in decoders.items()}\n\n\ndef layouts(reps: int = 20, world: World = World()) -> pd.DataFrame:\n    \"\"\"Exp 1: oracle prompt groups, test layouts from 1 to 18 authors per prompt.\"\"\"\n    rng = np.random.default_rng(SEED)\n    rows = []\n    for m in LAYOUTS:\n        for r in range(reps):\n            pool = test_pool(world, N_TEST, m, rng)\n            for name, acc in _score(pool, oracle(pool.content, pool.prompts), world).items():\n                rows.append(dict(m=str(m), rep=r, decoder=name, bal_acc=acc))\n    return pd.DataFrame(rows)\n\n\ndef grouping(reps: int = 10, ms=(2, 18, \"mixed\"), etas=(0.3, 0.6, 0.9), thresholds=(0.3, 0.5, 0.7)) -> pd.DataFrame:\n    \"\"\"Exp 2: recovered groups (content clustering) of varying purity, against oracle groups and none.\"\"\"\n    rng = np.random.default_rng(SEED + 1)\n    rows = []\n    for eta in etas:\n        w = World(eta=eta)\n        for m in ms:\n            for r in range(reps):\n                pool = test_pool(w, N_TEST, m, rng)\n                groupings = {\"none\": singletons(pool.content, pool.prompts), \"oracle\": oracle(pool.content, pool.prompts)}\n                for t in thresholds:\n                    groupings[f\"cluster@{t}\"] = cluster(pool.content, t)\n                for gname, g in groupings.items():\n                    prec, rec = pair_quality(g, pool.prompts)\n                    for name, acc in _score(pool, g, w).items():\n                        rows.append(dict(eta=eta, m=str(m), rep=r, grouping=gname, precision=prec, recall=rec,\n                                         decoder=name, bal_acc=acc))\n    return pd.DataFrame(rows)\n\n\ndef prior_shift(reps: int = 20, strengths=(0.0, 0.5, 1.0, 2.0), m: int = 2, world: World = World()) -> pd.DataFrame:\n    \"\"\"Exp 3: unequal author frequencies in the test pool; balancing assumes they are equal. The classifier is\n    either unbiased (its scores are the likelihood) or carries the prior of an unbalanced training pool.\"\"\"\n    rng = np.random.default_rng(SEED + 2)\n    train_prior = skewed_prior(1.0, np.random.default_rng(SEED + 20))  # fixed, like a pool of 1,100-3,300 per author\n    rows = []\n    for st in strengths:\n        for r in range(reps):\n            prior = skewed_prior(st, rng)\n            pool = sample(world, *layout(N_TEST // m, m, rng, prior), rng)\n            ll = _loglik(pool.scores, world.mu, world.sigma**2 + world.tau**2)\n            tv = 0.5 * np.abs(prior - 1 / K).sum()\n            for bias, z in [(\"none\", ll), (\"train prior\", ll + np.log(train_prior))]:\n                preds = {\"plain\": z.argmax(1), \"sinkhorn_uniform\": sinkhorn(z).argmax(1),\n                         \"sinkhorn_true_prior\": sinkhorn(z, prior).argmax(1)}\n                for name, p in preds.items():\n                    rows.append(dict(strength=st, rep=r, prior_tv=tv, classifier_bias=bias, decoder=name,\n                                     bal_acc=balanced_accuracy(pool.labels, p)))\n    return pd.DataFrame(rows)\n\n\ndef promise(reps: int = 20, world: World = World(eta=0.6), threshold: float = 0.5) -> pd.DataFrame:\n    \"\"\"Exp 4: the gain a train-layout CV promises (150 prompts x 18 authors, as in the competition's train)\n    against the gain realised on test pools with fewer authors per prompt, with the same clustering.\"\"\"\n    rng = np.random.default_rng(SEED + 3)\n    rows = []\n    for r in range(reps):\n        settings = {\"cv (18 per prompt, 150 prompts)\": sample(world, *layout(150, 18, rng), rng)}\n        for m in [1, 2, 3, \"mixed\"]:\n            settings[f\"test ({m} per prompt)\"] = test_pool(world, N_TEST, m, rng)\n        for sname, pool in settings.items():\n            g = cluster(pool.content, threshold)\n            for name, acc in _score(pool, g, world).items():\n                rows.append(dict(setting=sname, rep=r, decoder=name, bal_acc=acc))\n    return pd.DataFrame(rows)\n\n\n# ----------------------------------------------------------------------------------------------- summaries\n\ndef _gain(df: pd.DataFrame, keys: list[str]) -> pd.DataFrame:\n    base = df[df.decoder == \"plain\"].set_index(keys + [\"rep\"])[\"bal_acc\"]\n    d = df.join(base.rename(\"plain_acc\"), on=keys + [\"rep\"])\n    d[\"gain\"] = d.bal_acc - d.plain_acc\n    out = d.groupby(keys + [\"decoder\"]).agg(bal_acc=(\"bal_acc\", \"mean\"), gain=(\"gain\", \"mean\"),\n                                           gain_sd=(\"gain\", \"std\")).reset_index()\n    return out\n\n\ndef summarise_layouts(df):\n    t = _gain(df, [\"m\"]).pivot(index=\"m\", columns=\"decoder\", values=\"gain\")\n    t.insert(0, \"plain_acc\", df[df.decoder == \"plain\"].groupby(\"m\").bal_acc.mean())\n    return t.reindex([str(m) for m in LAYOUTS]).round(4).reset_index()\n\n\ndef summarise_grouping(df):\n    q = df.groupby([\"eta\", \"m\", \"grouping\"])[[\"precision\", \"recall\"]].mean()\n    t = _gain(df, [\"eta\", \"m\", \"grouping\"]).pivot(index=[\"eta\", \"m\", \"grouping\"], columns=\"decoder\", values=\"gain\")\n    return q.join(t).round(4).reset_index()\n\n\ndef summarise_prior(df):\n    t = df.groupby([\"classifier_bias\", \"strength\", \"decoder\"]).bal_acc.mean().unstack()\n    t = t.join(df.groupby([\"classifier_bias\", \"strength\"]).prior_tv.mean())\n    return t[[\"prior_tv\", \"plain\", \"sinkhorn_uniform\", \"sinkhorn_true_prior\"]].round(4).reset_index()\n\n\ndef summarise_promise(df):\n    return _gain(df, [\"setting\"]).pivot(index=\"setting\", columns=\"decoder\", values=\"gain\").round(4).reset_index()\n",
"pgbench/methods.py": "\"\"\"Test-time decoders for the scores of sim.py, and ways to recover the prompt groups.\n\nEvery decoder takes the score matrix and a group id per passage (singletons have their own id) and\nreturns predicted labels. They know the world's parameters (mu, tau, sigma): the point is to measure what\neach decision rule can gain under a given test layout, not to estimate the parameters.\n\"\"\"\n\nfrom __future__ import annotations\n\nimport numpy as np\nfrom scipy.cluster.hierarchy import fcluster, linkage\nfrom scipy.optimize import linear_sum_assignment\n\nfrom .sim import K, World\n\n\ndef _loglik(s: np.ndarray, mu: np.ndarray, var: float) -> np.ndarray:\n    \"\"\"log N(s; mu_y, var I) per class, up to a constant: shape (n, K).\"\"\"\n    return -((s[:, None, :] - mu[None]) ** 2).sum(-1) / (2 * var)\n\n\ndef _members(groups: np.ndarray) -> list[np.ndarray]:\n    order = np.argsort(groups, kind=\"stable\")\n    cuts = np.flatnonzero(np.diff(groups[order])) + 1\n    return [g for g in np.split(order, cuts) if len(g) > 1]\n\n\ndef plain(s: np.ndarray, groups: np.ndarray, w: World) -> np.ndarray:\n    \"\"\"Each passage on its own: the prompt pull is just extra noise.\"\"\"\n    return _loglik(s, w.mu, w.sigma**2 + w.tau**2).argmax(1)\n\n\ndef center(s: np.ndarray, groups: np.ndarray, w: World) -> np.ndarray:\n    \"\"\"Subtract the group mean of the scores, then decode as if the pull were gone (the usual heuristic).\"\"\"\n    s = s.copy()\n    for g in _members(groups):\n        s[g] -= s[g].mean(0)\n    mu_c = w.mu - w.mu.mean(0)  # the group mean also removes about this much of the author signal\n    return _loglik(s, mu_c, w.sigma**2).argmax(1)\n\n\ndef hungarian(s: np.ndarray, groups: np.ndarray, w: World) -> np.ndarray:\n    \"\"\"Plain log-likelihoods, then at most one passage per author inside each group.\"\"\"\n    ll = _loglik(s, w.mu, w.sigma**2 + w.tau**2)\n    pred = ll.argmax(1)\n    for g in _members(groups):\n        if len(g) <= K:\n            rows, cols = linear_sum_assignment(-ll[g])\n            pred[g[rows]] = cols\n    return pred\n\n\ndef joint(s: np.ndarray, groups: np.ndarray, w: World, no_repeat: bool = False, sweeps: int = 4) -> np.ndarray:\n    \"\"\"Approximate joint MAP under the model: each member is decoded against the prompt pull estimated\n    from the other members' residuals (iterated conditional modes); optionally one passage per author.\"\"\"\n    t2, s2 = w.tau**2, w.sigma**2\n    pred = plain(s, groups, w)\n    for g in _members(groups):\n        m = len(g)\n        sg, yg = s[g], pred[g].copy()\n        for _ in range(sweeps):\n            r = sg - w.mu[yg]\n            tot = r.sum(0)\n            # Posterior of the pull given the other m-1 members, then the predictive of member j.\n            shrink = t2 / (s2 + (m - 1) * t2)\n            u_hat = shrink * (tot[None] - r)\n            var = s2 + t2 * s2 / (s2 + (m - 1) * t2)\n            ll = -((sg[:, None, :] - u_hat[:, None, :] - w.mu[None]) ** 2).sum(-1) / (2 * var)\n            if no_repeat and m <= K:\n                rows, cols = linear_sum_assignment(-ll)\n                new = np.empty(m, int)\n                new[rows] = cols\n            else:\n                new = ll.argmax(1)\n            if np.array_equal(new, yg):\n                break\n            yg = new\n        pred[g] = yg\n    return pred\n\n\ndef sinkhorn(logp: np.ndarray, target: np.ndarray | None = None, iters: int = 100) -> np.ndarray:\n    \"\"\"Rescale class columns until each class holds the target share of the probability mass.\"\"\"\n    p = np.exp(logp - logp.max(1, keepdims=True))\n    p /= p.sum(1, keepdims=True)\n    t = np.full(p.shape[1], 1 / p.shape[1]) if target is None else target\n    for _ in range(iters):\n        p *= (t * len(p) / p.sum(0))[None]\n        p /= p.sum(1, keepdims=True)\n    return p\n\n\ndef plain_balanced(s: np.ndarray, groups: np.ndarray, w: World) -> np.ndarray:\n    return sinkhorn(_loglik(s, w.mu, w.sigma**2 + w.tau**2)).argmax(1)\n\n\nDECODERS = {\n    \"plain\": plain,\n    \"center\": center,\n    \"hungarian\": hungarian,\n    \"joint\": joint,\n    \"joint_no_repeat\": lambda s, g, w: joint(s, g, w, no_repeat=True),\n    \"plain_sinkhorn\": plain_balanced,\n}\n\n\n# ----------------------------------------------------------------------------------------------- grouping\n\ndef singletons(content: np.ndarray, prompts: np.ndarray) -> np.ndarray:\n    return np.arange(len(content))\n\n\ndef oracle(content: np.ndarray, prompts: np.ndarray) -> np.ndarray:\n    return prompts.copy()\n\n\ndef cluster(content: np.ndarray, threshold: float, max_size: int = K) -> np.ndarray:\n    \"\"\"Average-linkage clusters on cosine distance, cut at threshold; clusters above max_size become singletons.\"\"\"\n    z = linkage(content, method=\"average\", metric=\"cosine\")\n    g = fcluster(z, t=threshold, criterion=\"distance\")\n    sizes = np.bincount(g)\n    out = g.astype(np.int64)\n    big = sizes[g] > max_size\n    out[big] = g.max() + 1 + np.flatnonzero(big)\n    return out\n\n\ndef pair_quality(groups: np.ndarray, prompts: np.ndarray) -> tuple[float, float]:\n    \"\"\"Precision and recall of same-group pairs as same-prompt pairs.\"\"\"\n    same_g = groups[:, None] == groups[None, :]\n    same_p = prompts[:, None] == prompts[None, :]\n    off = ~np.eye(len(groups), dtype=bool)\n    tp = (same_g & same_p & off).sum()\n    pg, pp = (same_g & off).sum(), (same_p & off).sum()\n    return (tp / pg if pg else 1.0), (tp / pp if pp else 1.0)\n",
"pgbench/sim.py": "\"\"\"A known-truth model of authorship attribution when every passage answers a prompt.\n\nEach passage i, written by author y_i (one of K = 18) in answer to prompt p_i, gets a score vector\n\n    s_i = mu[y_i] + u[p_i] + e_i,      u[p] ~ N(0, tau^2 I),  e_i ~ N(0, sigma^2 I)\n\nmu[y] = beta * onehot(y) + beta_family * onehot-of-family(y) spread over the family's classes, so the family\n(GPT, Gemini, Claude, GLM, Qwen, Kimi) is easy and the author within a family is hard. u[p] is the prompt's\npull: every answer to the same prompt shares it, which is what test-time grouping tries to remove. Each\npassage also gets a content vector c_i = v[p_i] + eta * noise, which is all a grouping method can see.\n\nTrain has the competition's layout (every prompt answered by all 18 authors once). The test layout is a\nparameter: m authors per prompt, drawn without replacement, optionally with unequal author frequencies.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\n\nimport numpy as np\n\nFAMILIES = [5, 4, 5, 2, 1, 1]  # GPT, Gemini, Claude, GLM, Qwen, Kimi\nK = sum(FAMILIES)\nFAMILY = np.repeat(np.arange(len(FAMILIES)), FAMILIES)\n\n\n@dataclass(frozen=True)\nclass World:\n    beta: float = 1.0          # author signal\n    beta_family: float = 2.0   # family signal, shared by the authors of a family\n    tau: float = 0.7           # prompt pull (sd per score dimension)\n    sigma: float = 1.0         # passage noise\n    content_dim: int = 32\n    eta: float = 0.6           # content noise: higher means prompts are harder to recover\n\n    @property\n    def mu(self) -> np.ndarray:\n        fam = (FAMILY[:, None] == FAMILY[None, :]).astype(float)\n        return self.beta * np.eye(K) + self.beta_family * fam / fam.sum(1, keepdims=True)\n\n\n@dataclass\nclass Pool:\n    scores: np.ndarray    # (n, K)\n    labels: np.ndarray    # (n,)\n    prompts: np.ndarray   # (n,)\n    content: np.ndarray   # (n, content_dim), unit rows\n\n\ndef layout(n_prompts: int, m: int | np.ndarray, rng: np.random.Generator, prior: np.ndarray | None = None):\n    \"\"\"Labels and prompt ids: prompt p gets m (or m[p]) distinct authors, drawn with probability prior.\"\"\"\n    ms = np.broadcast_to(np.asarray(m), (n_prompts,))\n    p = np.full(K, 1 / K) if prior is None else prior\n    labels, prompts = [], []\n    for j, mj in enumerate(ms):\n        labels.append(rng.choice(K, size=int(mj), replace=False, p=p))\n        prompts.append(np.full(int(mj), j))\n    return np.concatenate(labels), np.concatenate(prompts)\n\n\ndef sample(world: World, labels: np.ndarray, prompts: np.ndarray, rng: np.random.Generator) -> Pool:\n    n_prompts = prompts.max() + 1\n    u = rng.normal(0, world.tau, (n_prompts, K))\n    s = world.mu[labels] + u[prompts] + rng.normal(0, world.sigma, (len(labels), K))\n    v = rng.normal(size=(n_prompts, world.content_dim))\n    v /= np.linalg.norm(v, axis=1, keepdims=True)\n    c = v[prompts] + world.eta * rng.normal(size=(len(labels), world.content_dim)) / np.sqrt(world.content_dim)\n    c /= np.linalg.norm(c, axis=1, keepdims=True)\n    return Pool(s, labels, prompts, c)\n\n\ndef test_pool(world: World, n: int, m: int | str, rng: np.random.Generator, prior: np.ndarray | None = None) -> Pool:\n    \"\"\"About n test passages with m authors per prompt; m=\"mixed\" draws m from {1, 2, 3, 6, 18}.\"\"\"\n    if m == \"mixed\":\n        ms = rng.choice([1, 2, 3, 6, 18], size=n, p=[0.35, 0.25, 0.15, 0.15, 0.10])\n        ms = ms[: np.searchsorted(np.cumsum(ms), n) + 1]\n        return sample(world, *layout(len(ms), ms, rng, prior), rng)\n    return sample(world, *layout(max(1, n // int(m)), int(m), rng, prior), rng)\n\n\ndef skewed_prior(strength: float, rng: np.random.Generator) -> np.ndarray:\n    \"\"\"Unequal author frequencies: a Dirichlet draw that is more uneven for larger strength; 0 gives uniform.\"\"\"\n    if strength <= 0:\n        return np.full(K, 1 / K)\n    p = rng.dirichlet(np.full(K, 1 / strength))\n    return 0.5 * p + 0.5 / K  # every author keeps at least half the uniform share\n\n\ndef balanced_accuracy(y: np.ndarray, pred: np.ndarray) -> float:\n    return float(np.mean([np.mean(pred[y == k] == k) for k in np.unique(y)]))\n"
}

import os, sys
ROOT = "/kaggle/working/prompt-grouping-bench" if os.path.isdir("/kaggle/working") else os.path.abspath("prompt-grouping-bench")
for path, text in SOURCES.items():
    os.makedirs(os.path.dirname(os.path.join(ROOT, path)), exist_ok=True)
    with open(os.path.join(ROOT, path), "w", encoding="utf-8") as f:
        f.write(text)
sys.path.insert(0, ROOT)


In [ ]:
import time, warnings
warnings.simplefilter("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pgbench import experiments as ex
pd.set_option("display.precision", 4); pd.set_option("display.width", 160)


## 1. How many answers per prompt does grouping need?

The test pool has `m` authors per prompt and the decoders get the true prompt groups. Each cell is the mean gain in balanced accuracy over `plain` (20 pools of 4,000 passages per layout). `mixed` draws m from {1, 2, 3, 6, 18} per prompt.

In [ ]:
t = time.time(); e1 = ex.layouts(); print(round(time.time() - t), "s")
s1 = ex.summarise_layouts(e1); s1


In [ ]:
order = [m for m in s1.m if m != "mixed"]
x = [int(m) for m in order]
fig, ax = plt.subplots(figsize=(8, 4.5))
for name in ["center", "hungarian", "joint", "joint_no_repeat", "plain_sinkhorn"]:
    ax.plot(x, s1.set_index("m").loc[order, name], marker="o", label=name)
ax.axhline(0, color="k", lw=0.8); ax.set_xscale("log"); ax.set_xticks(x); ax.set_xticklabels(order)
ax.set_xlabel("authors per prompt in the test pool (oracle groups)"); ax.set_ylabel("gain in balanced accuracy over plain")
ax.legend(); plt.tight_layout(); plt.show()


## 2. Recovered groups instead of true ones

The groups now come from average-linkage clustering of the content vectors, cut at 0.3, 0.5 or 0.7 (clusters above 18 passages are split back into singletons). `eta` is the content noise: higher means prompts are harder to recover. `precision` and `recall` are for same-group pairs as same-prompt pairs. 10 pools per cell.

In [ ]:
t = time.time(); e2 = ex.grouping(); print(round(time.time() - t), "s")
s2 = ex.summarise_grouping(e2); s2


## 3. Balancing when authors are not equally frequent

Sinkhorn balancing pushes every author to the same share of the probability mass. Here the test pool's author frequencies are uneven (`prior_tv` is their total-variation distance from uniform; 2 authors per prompt). The classifier is either unbiased or carries the prior of an unbalanced training pool (`train prior`), as a model trained on a public pool of 1,100 to 3,300 passages per author would.

In [ ]:
t = time.time(); e3 = ex.prior_shift(); print(round(time.time() - t), "s")
ex.summarise_prior(e3)


## 4. What a train-layout CV promises

The competition's train has every prompt answered by all 18 authors. A grouped CV on that layout measures grouping gains at m = 18. Here is that promise next to the gain on test pools with fewer answers per prompt, with the same clustering (cut 0.5).

In [ ]:
t = time.time(); e4 = ex.promise(); print(round(time.time() - t), "s")
ex.summarise_promise(e4)


## 5. The pools in the dataset

The companion dataset holds one simulated test pool per layout, with every passage's true author, prompt and content vector.

In [ ]:
import glob
from pgbench.methods import DECODERS, oracle, cluster
from pgbench.sim import World, Pool, balanced_accuracy
hits = glob.glob("/kaggle/input/**/test_pools.csv", recursive=True)
if hits:
    df = pd.read_csv(hits[0]); w = World(); rows = []
    for m, g in df.groupby("layout", sort=False):
        pool = Pool(g.filter(like="score_").to_numpy(), g.author.to_numpy(), g.prompt.to_numpy(), g.filter(like="content_").to_numpy())
        for gname, groups in [("oracle", oracle(pool.content, pool.prompts)), ("cluster@0.5", cluster(pool.content, 0.5))]:
            rows.append({"layout": m, "grouping": gname, **{k: balanced_accuracy(pool.labels, f(pool.scores, groups, w)) for k, f in DECODERS.items()}})
    display(pd.DataFrame(rows))
else:
    print("dataset not attached")


## Findings

Numbers are from the tables above (plain decoding reaches about 0.353 balanced accuracy in every setting).

**1. The gain depends on how many answers a prompt really has.** With the true groups, at 18 answers per prompt
Hungarian assignment adds +0.117 and centring +0.101. At 2 answers per prompt Hungarian adds only +0.006 and centring
*loses* 0.067: subtracting a two-passage mean removes half of each passage's own author signal along with the prompt
effect. At 3 answers centring is back to zero. Joint decoding, which estimates the prompt effect from the *other*
members, is never worse than plain with true groups (+0.024 at 2 answers, +0.115 with the constraint at 18).

**2. Wrong merges turn the gains negative.** With recovered groups, what matters is pair precision. At 2 answers per
prompt and a loose cut (precision 0.14), Hungarian loses 0.017 and joint decoding with the constraint 0.022; centring
loses at every cut. The best cut differs by layout: at 18 answers per prompt a tight cut (0.3) misses pairs (recall
0.82, +0.096 for Hungarian instead of +0.122 at cut 0.5), while at 2 answers the cut of 0.5 that is best for the
train-like layout merges too much (precision 0.59). With noisier content (eta 0.9) and 2 answers, the 0.5 cut has
precision 0.33 and every grouping decoder is at or below plain (centring -0.081).

**3. A train-layout CV over-promises.** The competition's train answers every prompt 18 times, so a grouped CV
measures the gain at 18 answers: +0.118 for Hungarian, +0.111 for joint with the constraint, +0.098 for centring. On
test pools with one answer per prompt, the same clustering yields -0.004, -0.019 and -0.102; with two answers +0.004,
+0.015 and -0.057. This is the pattern reported on the forum (a CV gain of about +0.10 that became a small loss on
the public board), and it follows from the layout alone, without any modelling error.

**4. Balance to uniform, never to the true prior, for balanced accuracy.** Balanced accuracy weights every author's
recall equally, so the best decision ignores the author frequencies. For a classifier whose scores are the
likelihood, Sinkhorn balancing to equal mass costs about 0.003 at most, even when the test frequencies are far from equal
(total variation 0.23); balancing to the *true* uneven frequencies costs up to 0.040. For a classifier that carries an
unbalanced training prior, balancing to equal mass recovers +0.020 to +0.022 and comes within about 0.003 of the unbiased
classifier, whatever the test frequencies. So balancing removes a training-prior bias; it is not a bet that the test
is balanced.

**What this means for a real pipeline.** Measure grouping tricks with a CV that subsamples each validation prompt to
the number of answers the test is likely to have (1-3, if the forum reports are right), with the groups recovered the
same way as on the test, and prefer decoders that estimate the shared effect from the other members over plain
centring.

**Limitations.** Gaussian scores with an isotropic prompt effect; decoders that know the true parameters; one content
representation. The sizes are illustrative, the directions are the point.
